# Nsight Systems Training Notebook: CLIP ViT-B/32 Inference on Google Colab

This notebook is a practical training notebook for reading **Nsight Systems** traces using **CLIP ViT-B/32 inference** as the target workload.

The goal is not to maximize CLIP accuracy. The goal is to learn how to read the GPU execution timeline:

- CPU-side preprocessing
- image batch transfer to GPU
- CLIP vision encoder inference
- CUDA kernel launches
- GPU idle gaps
- effect of batch size
- warmup vs measured iterations
- NVTX ranges
- `nsys profile`
- `nsys stats`
- optional SQLite export and analysis

Recommended Colab runtime:

```text
Runtime type: GPU
GPU: T4 / L4 / A100 if available
```

This notebook is intentionally structured around **inference**, because VLA systems often require low-latency image-to-action or image-language-action execution.

## 0. What you should learn from this notebook

When reading an Nsight Systems trace, do not start by asking only:

```text
Which kernel is slow?
```

Start with a higher-level execution question:

```text
Is the GPU continuously busy?
Is Python/CPU feeding the GPU efficiently?
Is data transfer visible?
Does batch size change utilization?
Are there unexpected synchronization points?
Can I map model stages to timeline regions?
```

For VLA-style systems, this maps naturally to:

```text
camera/image preprocessing
  ↓
vision encoder
  ↓
language/action module
  ↓
robot action output
```

This notebook focuses on the first major component:

```text
image preprocessing + CLIP ViT-B/32 vision encoder inference
```

## 1. Environment check

First, check whether Colab has a GPU and whether `nsys` is already installed.

In some Colab runtimes, `nsys` may already exist under CUDA tools. In others, it may need installation. Colab environments are ephemeral, so installation may need to be repeated after runtime reset.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

def run(cmd, check=False):
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")
    return result

run("nvidia-smi")
run("which nsys || true")
run("nsys --version || true")

## 2. Install Python dependencies

We use:

- `torch`
- `torchvision`
- `transformers`
- `Pillow`
- `pandas`

The model is `openai/clip-vit-base-patch32` from Hugging Face Transformers.

Why this model?

```text
CLIP ViT-B/32 is small enough for Colab.
It has a real ViT vision encoder.
It is directly relevant to VLM/VLA-style pipelines.
It creates meaningful CUDA activity without requiring a giant model.
```

In [ ]:
# Colab usually already has torch installed.
# This cell keeps installation explicit and reproducible.
run("python -m pip install -q --upgrade transformers pillow pandas", check=True)

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 3. Create a small benchmark script

Nsight Systems works best when profiling a standalone Python script.

This cell writes `clip_vit_b32_infer_profile.py`.

The script supports:

```text
--batch-size
--num-iters
--warmup-iters
--dtype fp32/fp16/bf16
--use-nvtx
--include-preprocess
--profile-text
```

Important design choices:

- The model is loaded once.
- Warmup iterations are run before measured iterations.
- CUDA synchronization is placed around measured regions so timing is meaningful.
- NVTX ranges are added so the Nsight timeline has readable semantic blocks.
- Synthetic images are used by default to avoid network/image download variability.

In [ ]:
%%writefile clip_vit_b32_infer_profile.py
import argparse
import time
from contextlib import nullcontext

import torch
from PIL import Image
import numpy as np
from transformers import CLIPProcessor, CLIPModel


def nvtx_range(name: str, enabled: bool):
    if enabled and torch.cuda.is_available():
        return torch.cuda.nvtx.range(name)
    return nullcontext()


def make_synthetic_pil_images(batch_size: int, image_size: int = 224):
    # Synthetic images keep the benchmark deterministic and avoid network/file I/O.
    images = []
    rng = np.random.default_rng(42)
    for _ in range(batch_size):
        arr = rng.integers(0, 256, size=(image_size, image_size, 3), dtype=np.uint8)
        images.append(Image.fromarray(arr, mode="RGB"))
    return images


def parse_dtype(dtype_name: str):
    if dtype_name == "fp32":
        return torch.float32
    if dtype_name == "fp16":
        return torch.float16
    if dtype_name == "bf16":
        return torch.bfloat16
    raise ValueError(f"Unsupported dtype: {dtype_name}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--num-iters", type=int, default=20)
    parser.add_argument("--warmup-iters", type=int, default=5)
    parser.add_argument("--dtype", type=str, default="fp16", choices=["fp32", "fp16", "bf16"])
    parser.add_argument("--use-nvtx", action="store_true")
    parser.add_argument("--include-preprocess", action="store_true")
    parser.add_argument("--profile-text", action="store_true")
    args = parser.parse_args()

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Please enable a GPU runtime.")

    device = torch.device("cuda")
    dtype = parse_dtype(args.dtype)

    print("device:", torch.cuda.get_device_name(0))
    print("batch_size:", args.batch_size)
    print("dtype:", dtype)
    print("num_iters:", args.num_iters)
    print("warmup_iters:", args.warmup_iters)
    print("include_preprocess:", args.include_preprocess)
    print("profile_text:", args.profile_text)

    with nvtx_range("load_model_and_processor", args.use_nvtx):
        model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        model.eval().to(device)

        # fp16/bf16 primarily affects model computation.
        # CLIP preprocessing still produces fp32 pixel_values unless converted.
        if dtype in (torch.float16, torch.bfloat16):
            model = model.to(dtype=dtype)

    images = make_synthetic_pil_images(args.batch_size)

    # Prepare text once when requested. This gives a tiny VLM-like flavor.
    texts = ["a photo of a robot"] * args.batch_size

    with nvtx_range("initial_preprocess", args.use_nvtx):
        if args.profile_text:
            inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
        else:
            inputs = processor(images=images, return_tensors="pt")

        pixel_values = inputs["pixel_values"].to(device=device, dtype=dtype, non_blocking=True)
        if args.profile_text:
            input_ids = inputs["input_ids"].to(device=device, non_blocking=True)
            attention_mask = inputs["attention_mask"].to(device=device, non_blocking=True)
        else:
            input_ids = None
            attention_mask = None

    # Warmup: important for GPU frequency, CUDA lazy initialization, and framework caches.
    with torch.inference_mode():
        for i in range(args.warmup_iters):
            with nvtx_range(f"warmup_iter_{i}", args.use_nvtx):
                if args.profile_text:
                    _ = model(
                        pixel_values=pixel_values,
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                    )
                else:
                    _ = model.get_image_features(pixel_values=pixel_values)

        torch.cuda.synchronize()

    iter_times_ms = []

    with torch.inference_mode():
        for i in range(args.num_iters):
            if args.include_preprocess:
                # This intentionally includes CPU preprocessing and H2D transfer.
                with nvtx_range(f"iter_{i}_preprocess_cpu", args.use_nvtx):
                    if args.profile_text:
                        inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
                    else:
                        inputs = processor(images=images, return_tensors="pt")

                with nvtx_range(f"iter_{i}_h2d_transfer", args.use_nvtx):
                    pixel_values = inputs["pixel_values"].to(device=device, dtype=dtype, non_blocking=True)
                    if args.profile_text:
                        input_ids = inputs["input_ids"].to(device=device, non_blocking=True)
                        attention_mask = inputs["attention_mask"].to(device=device, non_blocking=True)

            torch.cuda.synchronize()
            start = time.perf_counter()

            with nvtx_range(f"iter_{i}_clip_forward", args.use_nvtx):
                if args.profile_text:
                    output = model(
                        pixel_values=pixel_values,
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                    )
                    # Touch logits to prevent overly aggressive elimination assumptions.
                    y = output.logits_per_image
                else:
                    y = model.get_image_features(pixel_values=pixel_values)

            torch.cuda.synchronize()
            end = time.perf_counter()

            iter_times_ms.append((end - start) * 1000.0)

    avg_ms = sum(iter_times_ms) / len(iter_times_ms)
    print(f"avg_forward_latency_ms={avg_ms:.3f}")
    print(f"throughput_images_per_sec={args.batch_size * 1000.0 / avg_ms:.2f}")
    print("first_5_iter_times_ms:", [round(x, 3) for x in iter_times_ms[:5]])


if __name__ == "__main__":
    main()

## 4. Smoke test without Nsight Systems

Before profiling, make sure the script works normally.

Start with a small batch size. The first run may download the model and can take longer.

In [ ]:
run("python clip_vit_b32_infer_profile.py --batch-size 4 --num-iters 5 --warmup-iters 2 --dtype fp16 --use-nvtx", check=True)

## 5. Profile with Nsight Systems

Now run `nsys profile`.

Key options:

```text
--trace=cuda,nvtx,osrt
```

- `cuda`: CUDA API and kernel activity
- `nvtx`: named ranges from the script
- `osrt`: OS runtime calls, useful for CPU-side behavior

```text
--stats=true
```

prints summary tables after profiling.

```text
-o report_clip_bs8
```

writes `report_clip_bs8.nsys-rep`.

The report can be downloaded and opened in the local Nsight Systems GUI.

In [ ]:
run("""nsys profile \
  --trace=cuda,nvtx,osrt \
  --stats=true \
  --force-overwrite=true \
  -o report_clip_bs8 \
  python clip_vit_b32_infer_profile.py \
    --batch-size 8 \
    --num-iters 20 \
    --warmup-iters 5 \
    --dtype fp16 \
    --use-nvtx""", check=False)

run("ls -lh report_clip_bs8* || true")

## 6. Read the first report using `nsys stats`

The first reading exercise is not the GUI.

Start with `nsys stats` and inspect the tables.

Look for:

```text
CUDA API Summary
CUDA Kernel Summary
CUDA Memory Operation Summary
NVTX Summary
OS Runtime Summary
```

Questions to ask:

```text
Which CUDA kernels dominate time?
Are there many small kernels?
How much time is spent in cudaMemcpy/cudaMemcpyAsync?
Does the NVTX summary match the intended forward-pass regions?
Are there obvious synchronization calls?
```

In [ ]:
run("nsys stats report_clip_bs8.nsys-rep || true")

## 7. Compare batch sizes

This is the most important exercise.

Run the same CLIP vision encoder with different batch sizes:

```text
B=1
B=8
B=32
```

Expected pattern:

```text
B=1:
  lower latency per request, often worse GPU utilization

B=8:
  better utilization, moderate latency

B=32:
  better throughput, possibly higher memory pressure
```

For real-time VLA/robotics, `B=1` latency matters.  
For offline data processing or training, throughput matters.

In [ ]:
batch_sizes = [1, 8, 32]

for bs in batch_sizes:
    out = f"report_clip_bs{bs}"
    cmd = f'''nsys profile \
      --trace=cuda,nvtx,osrt \
      --stats=true \
      --force-overwrite=true \
      -o {out} \
      python clip_vit_b32_infer_profile.py \
        --batch-size {bs} \
        --num-iters 20 \
        --warmup-iters 5 \
        --dtype fp16 \
        --use-nvtx'''
    run(cmd, check=False)

run("ls -lh report_clip_bs*.nsys-rep || true")

## 8. Export reports to SQLite

Nsight Systems reports can be exported to SQLite.

This is useful in Colab because the GUI is not convenient there.

The schema can vary by Nsight Systems version, so the notebook first lists available tables.

In [ ]:
for bs in [1, 8, 32]:
    rep = f"report_clip_bs{bs}.nsys-rep"
    sqlite = f"report_clip_bs{bs}.sqlite"
    run(f"nsys export --type sqlite --force-overwrite=true --output {sqlite} {rep} || true")

run("ls -lh *.sqlite || true")

## 9. Inspect SQLite tables

This cell prints available tables for one report.

Useful table names often include items related to:

```text
CUDA API
CUDA kernel
CUDA memory operations
NVTX events
strings
```

Nsight Systems schema names can differ by version, so exploratory inspection is safer than hard-coding assumptions.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

sqlite_path = Path("report_clip_bs8.sqlite")

if sqlite_path.exists():
    con = sqlite3.connect(sqlite_path)
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
        con,
    )
    display(tables)
    con.close()
else:
    print("SQLite report not found. Run the export cell first.")

## 10. Try to summarize CUDA kernel-like tables

Different Nsight versions use slightly different schema names.

This helper searches for likely kernel tables and prints their columns.  
You can then decide which table to query.

In [ ]:
def show_candidate_tables(sqlite_file):
    con = sqlite3.connect(sqlite_file)
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
        con,
    )["name"].tolist()

    candidates = [
        t for t in tables
        if any(k.lower() in t.lower() for k in ["kernel", "cuda", "gpu", "memcpy", "nvtx"])
    ]

    print("Candidate tables:")
    for t in candidates:
        print("-", t)
        try:
            cols = pd.read_sql_query(f"PRAGMA table_info('{t}')", con)
            print("  columns:", cols["name"].tolist())
        except Exception as e:
            print("  could not read columns:", e)
    con.close()

if Path("report_clip_bs8.sqlite").exists():
    show_candidate_tables("report_clip_bs8.sqlite")
else:
    print("SQLite report not found.")

## 11. Optional: use `nsys stats` reports as CSV-like text

In many cases, `nsys stats` is enough for Colab-based training.

Run stats for all batch sizes and save text files.

In [ ]:
for bs in [1, 8, 32]:
    cmd = f"nsys stats report_clip_bs{bs}.nsys-rep > stats_clip_bs{bs}.txt || true"
    run(cmd)

run("ls -lh stats_clip_bs*.txt || true")

for bs in [1, 8, 32]:
    path = Path(f"stats_clip_bs{bs}.txt")
    if path.exists():
        print("=" * 80)
        print(f"stats_clip_bs{bs}.txt")
        print("=" * 80)
        print(path.read_text(errors="ignore")[:4000])

## 12. Profile with preprocessing included

Now intentionally include CPU preprocessing and host-to-device transfer inside each iteration.

This is important for VLA systems because real inference often includes:

```text
camera frame
image resize/crop/normalize
CPU-to-GPU transfer
vision encoder
policy/action head
```

Compare this report with the earlier one that reuses preprocessed GPU tensors.

In [ ]:
run("""nsys profile \
  --trace=cuda,nvtx,osrt \
  --stats=true \
  --force-overwrite=true \
  -o report_clip_bs8_with_preprocess \
  python clip_vit_b32_infer_profile.py \
    --batch-size 8 \
    --num-iters 20 \
    --warmup-iters 5 \
    --dtype fp16 \
    --use-nvtx \
    --include-preprocess""", check=False)

run("nsys stats report_clip_bs8_with_preprocess.nsys-rep || true")

## 13. Profile CLIP image + text path

This adds the CLIP text tower and logits computation.

This is still not a full VLA model, but it is closer to a VLM/VLA-style workload:

```text
image encoder
text encoder
image-text similarity/logits
```

In a real VLA, the language module may be much larger than this CLIP text encoder.

In [ ]:
run("""nsys profile \
  --trace=cuda,nvtx,osrt \
  --stats=true \
  --force-overwrite=true \
  -o report_clip_image_text_bs8 \
  python clip_vit_b32_infer_profile.py \
    --batch-size 8 \
    --num-iters 20 \
    --warmup-iters 5 \
    --dtype fp16 \
    --use-nvtx \
    --profile-text""", check=False)

run("nsys stats report_clip_image_text_bs8.nsys-rep || true")

## 14. Download reports for local Nsight Systems GUI

The best way to read the visual timeline is:

```text
1. Generate `.nsys-rep` in Colab
2. Download it
3. Open it with local Nsight Systems GUI
```

In the GUI, inspect:

```text
NVTX ranges
CUDA API row
CUDA kernel row
GPU utilization
Memcpy rows
CPU thread activity
idle gaps between kernels
```

In [ ]:
from google.colab import files
from pathlib import Path

reports = sorted(Path(".").glob("report_clip*.nsys-rep"))
print("Available reports:")
for p in reports:
    print(p)

# Uncomment one or more lines to download.
# files.download("report_clip_bs8.nsys-rep")
# files.download("report_clip_bs1.nsys-rep")
# files.download("report_clip_bs32.nsys-rep")
# files.download("report_clip_bs8_with_preprocess.nsys-rep")
# files.download("report_clip_image_text_bs8.nsys-rep")

## 15. Reading checklist for Nsight Systems GUI

When you open the report locally, use this checklist.

### A. NVTX ranges

Confirm that you can see ranges such as:

```text
load_model_and_processor
initial_preprocess
warmup_iter_*
iter_*_clip_forward
iter_*_preprocess_cpu
iter_*_h2d_transfer
```

These ranges are your semantic anchors.

### B. CUDA API row

Look for calls such as:

```text
cudaLaunchKernel
cudaMemcpyAsync
cudaDeviceSynchronize
cudaStreamSynchronize
```

Questions:

```text
Are synchronization calls frequent?
Are API calls densely packed or separated by CPU idle time?
```

### C. CUDA kernel row

Look for:

```text
many small kernels
long GEMM/attention-related kernels
layer norm / elementwise kernels
softmax-related kernels
```

Questions:

```text
Is GPU execution continuous?
Are kernels tiny and fragmented?
Does larger batch size make kernels longer and reduce idle gaps?
```

### D. Memory operations

Look for:

```text
Host to Device copies
Device to Host copies
Device to Device copies
```

Questions:

```text
Is preprocessing causing repeated H2D copies?
Is D2H caused by printing, logging, or converting tensors to NumPy?
```

### E. Batch-size comparison

Compare:

```text
report_clip_bs1
report_clip_bs8
report_clip_bs32
```

Expected interpretation:

```text
B=1:
  robotics-like single observation latency
  more overhead-sensitive

B=8:
  better GPU occupancy
  good middle ground

B=32:
  throughput-oriented
  may hide latency problems
```

## 16. How this connects to VLA profiling

This CLIP exercise covers the vision side of a VLA stack.

A realistic VLA profiling ladder would be:

```text
1. CLIP / ViT vision encoder
2. small LLM decoder
3. vision encoder + projector + small LLM
4. action head
5. diffusion/action chunk decoder
6. full VLA-style pipeline
```

The same Nsight reading habits transfer directly:

```text
NVTX ranges define semantic stages.
CUDA rows reveal actual GPU execution.
Gaps reveal CPU/framework/data pipeline bottlenecks.
Batch size separates throughput vs latency behavior.
Preprocessing tells you whether the model is really the bottleneck.
```

For VLA/robotics, the most important distinction is:

```text
offline throughput is not the same as single-step control latency
```

So always profile both:

```text
batch-size 1 latency
batch-size N throughput
```

## 17. Suggested follow-up exercises

After this notebook, extend the script in these directions:

### Exercise 1: Replace CLIP with DINOv2 or SigLIP

Compare the CUDA timeline.

### Exercise 2: Add a small Transformer decoder

Use CLIP image features as prefix tokens and run a small decoder.

### Exercise 3: Add an action head

Map image features to:

```text
[B, action_dim]
```

or

```text
[B, horizon, action_dim]
```

### Exercise 4: Add a denoising loop

Create a small diffusion-policy-style action generator and inspect the repeated kernel pattern.

### Exercise 5: Add intentional bottlenecks

Examples:

```text
convert tensor to NumPy inside loop
move tensors CPU ↔ GPU each iteration
use batch size 1
add unnecessary torch.cuda.synchronize()
```

Then verify that Nsight Systems exposes the problem.